# 1C. Construcción del dataset: armado de la cohorte 1

Nota: requiere las carpetas de datos crudos de la API (`chartmetric_metadata*/`, `chartmetric_liveevents_2024_2025*/`) en la raíz del proyecto. No se incluyen en el repositorio por las restricciones de uso de Chartmetric.

In [63]:
import pandas as pd
import numpy as np

# Eventos en vivo agregados

In [64]:
ruta_eventlevel = r"chartmetric_liveevents_2024_2025\liveevents_2024_2025_eventlevel.csv"

In [65]:
df_events = pd.read_csv(ruta_eventlevel)

In [66]:
print(df_events.shape)
print(df_events.columns.tolist())
df_events.head(3)

(35810, 24)
['downloaded_at_utc', 'chartmetric_id', 'artist_name', 'event_id', 'event_name', 'type', 'event_year', 'start_date', 'end_date', 'venue_id', 'venue_name', 'venue_capacity', 'city_id', 'city_name', 'code2', 'price', 'low_price', 'high_price', 'currency', 'is_headliner', 'jambase_event_url', 'songkick_event_url', 'seatgeek_event_url', 'ticketmaster_event_url']


In [67]:
# fechas
df_events["start_date"] = pd.to_datetime(df_events["start_date"], errors="coerce")
df_events["end_date"] = pd.to_datetime(df_events["end_date"], errors="coerce")

# por seguridad, si event_year vino mal tipado
df_events["event_year"] = pd.to_numeric(df_events["event_year"], errors="coerce")

# columnas numéricas a usar
cols_numericas = ["low_price", "high_price", "venue_capacity"]

for col in cols_numericas:
    df_events[col] = pd.to_numeric(df_events[col], errors="coerce")

In [68]:
print(df_events["event_year"].value_counts(dropna=False).sort_index())

event_year
2024    14373
2025    21437
Name: count, dtype: int64


In [69]:
def moneda_principal(serie):
    serie_limpia = serie.dropna()
    if serie_limpia.empty:
        return np.nan
    return serie_limpia.mode().iloc[0]

In [70]:
def resumir_live_por_anio(df, anio):
    df_anio = df[df["event_year"] == anio].copy()
    
    resumen = (
        df_anio
        .groupby(["chartmetric_id", "artist_name"], as_index=False)
        .agg(
            n_shows=("event_id", "nunique"),
            avg_low_price=("low_price", "mean"),
            avg_high_price=("high_price", "mean"),
            avg_venue_capacity=("venue_capacity", "mean"),
            total_capacity=("venue_capacity", "sum"),
            n_shows_with_capacity=("venue_capacity", lambda x: x.notna().sum()),
            n_cities=("city_name", lambda x: x.dropna().nunique()),
            n_countries=("code2", lambda x: x.dropna().nunique()),
            n_currencies=("currency", lambda x: x.dropna().nunique()),
            main_currency=("currency", moneda_principal)
        )
    )
    
    resumen = resumen.rename(columns={
        "n_shows": f"n_shows_{anio}",
        "avg_low_price": f"avg_low_price_{anio}",
        "avg_high_price": f"avg_high_price_{anio}",
        "avg_venue_capacity": f"avg_venue_capacity_{anio}",
        "total_capacity": f"total_capacity_{anio}",
        "n_shows_with_capacity": f"n_shows_with_capacity_{anio}",
        "n_cities": f"n_cities_{anio}",
        "n_countries": f"n_countries_{anio}",
        "n_currencies": f"n_currencies_{anio}",
        "main_currency": f"main_currency_{anio}"
    })
    
    # si no hay ningún dato de capacidad, dejar total_capacity como NaN
    col_total = f"total_capacity_{anio}"
    col_ncap = f"n_shows_with_capacity_{anio}"
    resumen.loc[resumen[col_ncap] == 0, col_total] = np.nan
    
    return resumen

In [71]:
df_live_2024 = resumir_live_por_anio(df_events, 2024)
df_live_2025 = resumir_live_por_anio(df_events, 2025)

print("df_live_2024:", df_live_2024.shape)
print("df_live_2025:", df_live_2025.shape)

df_live_2024: (816, 12)
df_live_2025: (941, 12)


In [72]:
df_live_agg = df_live_2024.merge(
    df_live_2025,
    on=["chartmetric_id", "artist_name"],
    how="outer"
)

In [ ]:
# imputación de 0 para poder sumar las columnas bianuales. si hay NaN la suma queda NaN.


cols_cero = [
    "n_shows_2024",
    "n_cities_2024",
    "n_countries_2024",
    "n_currencies_2024",
    "n_shows_with_capacity_2024",
    "n_shows_2025",
    "n_cities_2025",
    "n_countries_2025",
    "n_currencies_2025",
    "n_shows_with_capacity_2025"
]

for col in cols_cero:
    if col in df_live_agg.columns:
        df_live_agg[col] = df_live_agg[col].fillna(0)

In [12]:
df_live_agg["shows_per_country_2025"] = np.where(
    df_live_agg["n_countries_2025"] > 0,
    df_live_agg["n_shows_2025"] / df_live_agg["n_countries_2025"],
    np.nan
)

df_live_agg["shows_per_country_2024"] = np.where(
    df_live_agg["n_countries_2024"] > 0,
    df_live_agg["n_shows_2024"] / df_live_agg["n_countries_2024"],
    np.nan
)

## Columnas bianuales

In [13]:
# columnas combinadas 2024 + 2025
df_live_agg["n_shows_total"] = (
    df_live_agg["n_shows_2024"] + df_live_agg["n_shows_2025"]
)

df_live_agg["n_shows_with_capacity_total"] = (
    df_live_agg["n_shows_with_capacity_2024"] + df_live_agg["n_shows_with_capacity_2025"]
)

df_live_agg["total_capacity_total"] = (
    df_live_agg["total_capacity_2024"].fillna(0) + df_live_agg["total_capacity_2025"].fillna(0)
)

# si no hubo ningún show con capacidad, dejar total_capacity_total como NaN
df_live_agg.loc[
    df_live_agg["n_shows_with_capacity_total"] == 0,
    "total_capacity_total"
] = np.nan

df_live_agg["avg_venue_capacity_total"] = np.where(
    df_live_agg["n_shows_with_capacity_total"] > 0,
    df_live_agg["total_capacity_total"] / df_live_agg["n_shows_with_capacity_total"],
    np.nan
)



In [14]:
def resumir_live_total(df):
    df_total = df[df["event_year"].isin([2024, 2025])].copy()
    
    resumen_total = (
        df_total
        .groupby(["chartmetric_id", "artist_name"], as_index=False)
        .agg(
            n_cities_total=("city_name", lambda x: x.dropna().nunique()),
            n_countries_total=("code2", lambda x: x.dropna().nunique())
        )
    )
    
    return resumen_total

In [15]:
df_live_total = resumir_live_total(df_events)

df_live_agg = df_live_agg.merge(
    df_live_total,
    on=["chartmetric_id", "artist_name"],
    how="left"
)

In [16]:
df_live_agg["shows_per_country_total"] = np.where(
    df_live_agg["n_countries_total"] > 0,
    df_live_agg["n_shows_total"] / df_live_agg["n_countries_total"],
    np.nan
)

In [17]:
print(df_live_agg.shape)
print(df_live_agg.columns.tolist())
df_live_agg.head(10)

(1004, 31)
['chartmetric_id', 'artist_name', 'n_shows_2024', 'avg_low_price_2024', 'avg_high_price_2024', 'avg_venue_capacity_2024', 'total_capacity_2024', 'n_shows_with_capacity_2024', 'n_cities_2024', 'n_countries_2024', 'n_currencies_2024', 'main_currency_2024', 'n_shows_2025', 'avg_low_price_2025', 'avg_high_price_2025', 'avg_venue_capacity_2025', 'total_capacity_2025', 'n_shows_with_capacity_2025', 'n_cities_2025', 'n_countries_2025', 'n_currencies_2025', 'main_currency_2025', 'shows_per_country_2025', 'shows_per_country_2024', 'n_shows_total', 'n_shows_with_capacity_total', 'total_capacity_total', 'avg_venue_capacity_total', 'n_cities_total', 'n_countries_total', 'shows_per_country_total']


In [18]:
print("Duplicados por chartmetric_id:", df_live_agg["chartmetric_id"].duplicated().sum())
print("Duplicados por artist_name:", df_live_agg["artist_name"].duplicated().sum())

Duplicados por chartmetric_id: 0
Duplicados por artist_name: 0


In [19]:
# reversión de los 0 asignados en las columnas de conteo live
# # que eran necesarios para crear las columnas bianuales

df_live_agg[cols_cero] = df_live_agg[cols_cero].replace(0, np.nan)

In [20]:
df_live_agg.isna().sum().sort_values(ascending=False)

avg_low_price_2024             414
avg_high_price_2024            414
n_currencies_2024              360
main_currency_2024             360
avg_low_price_2025             314
avg_high_price_2025            314
main_currency_2025             280
n_currencies_2025              280
total_capacity_2024            226
n_shows_with_capacity_2024     226
avg_venue_capacity_2024        226
n_countries_2024               194
shows_per_country_2024         194
n_cities_2024                  194
n_shows_2024                   188
n_shows_with_capacity_2025     107
total_capacity_2025            107
avg_venue_capacity_2025        107
n_countries_2025                73
shows_per_country_2025          73
n_cities_2025                   73
n_shows_2025                    63
total_capacity_total            29
avg_venue_capacity_total        29
shows_per_country_total          1
artist_name                      0
chartmetric_id                   0
n_shows_with_capacity_total      0
n_shows_total       

# Guardado de eventos agregados

In [21]:
ruta_salida_live_agg = r"chartmetric_liveevents_2024_2025\liveevents_2024_2025_artist_agg.csv"
df_live_agg.to_csv(ruta_salida_live_agg, index=False)

In [26]:
print("Artistas únicos en eventlevel:", df_events["chartmetric_id"].nunique())
print("Filas en df_live_agg:", df_live_agg.shape[0])

Artistas únicos en eventlevel: 1004
Filas en df_live_agg: 1004


In [27]:
df_live_summary = pd.read_csv(
    r"chartmetric_liveevents_2024_2025\liveevents_2024_2025_summary.csv"
)

In [29]:
ids_eventlevel = set(df_live_agg["chartmetric_id"].unique())
ids_summary = set(df_live_summary["chartmetric_id"].unique())

faltan_en_eventlevel = ids_summary - ids_eventlevel

print("Artistas en live_summary:", len(ids_summary))
print("Artistas en df_live_agg:", len(ids_eventlevel))
print("Artistas que están en summary pero no en eventlevel:", len(faltan_en_eventlevel))

Artistas en live_summary: 1400
Artistas en df_live_agg: 1004
Artistas que están en summary pero no en eventlevel: 396


In [ ]:
df_faltantes = df_live_summary[
    df_live_summary["chartmetric_id"].isin(faltan_en_eventlevel)
].copy()


(396, 13)


In [33]:
condicion_esperada = (
    (df_faltantes["ok"] == True) &
    (df_faltantes["status_code"] == 200) &
    (df_faltantes["n_shows_2024"] == 0) &
    (df_faltantes["n_shows_2025"] == 0)
)

print("Cantidad de faltantes que cumplen exactamente la condición esperada:", condicion_esperada.sum())
print("Total de faltantes:", len(df_faltantes))

Cantidad de faltantes que cumplen exactamente la condición esperada: 396
Total de faltantes: 396


# Incorporación de `band` al flat.csv

In [35]:
import os
import json
import pandas as pd

JSON_DIR = r"chartmetric_metadata\raw_json"
CSV_PATH = r"chartmetric_metadata\metadata_flat.csv"

# ------------------------------------------
# 1) extraer id + band desde JSON
# ------------------------------------------
rows_band = []

json_files = [f for f in os.listdir(JSON_DIR) if f.lower().endswith(".json")]

for fname in json_files:
    fpath = os.path.join(JSON_DIR, fname)

    try:
        with open(fpath, "r", encoding="utf-8") as f:
            data = json.load(f)

        obj = data.get("obj", data)

        rows_band.append({
            "response_id": obj.get("id"),   # ← clave alineada con el CSV
            "band": obj.get("band")
        })

    except Exception as e:
        print(f"Error leyendo {fname}: {e}")

df_band = pd.DataFrame(rows_band).drop_duplicates(subset=["response_id"])

# ------------------------------------------
# 2) leer metadata_flat
# ------------------------------------------
df_metadata = pd.read_csv(CSV_PATH)

# ------------------------------------------
# 3) merge limpio (sin tocar nada más)
# ------------------------------------------
df_metadata = df_metadata.drop(columns=["band"], errors="ignore")

df_metadata = df_metadata.merge(
    df_band,
    on="response_id",
    how="left"
)

# ------------------------------------------
# 4) guardar
# ------------------------------------------
df_metadata.to_csv(CSV_PATH, index=False)

print("metadata_flat.csv actualizado con la columna band")
print(df_metadata["band"].value_counts(dropna=False))

metadata_flat.csv actualizado con la columna band
band
False    1011
True      389
Name: count, dtype: int64


# Incorporación de variables adicionales del JSON

In [36]:
import os
import json
import pandas as pd

JSON_DIR = r"chartmetric_metadata\raw_json"
CSV_PATH = r"chartmetric_metadata\metadata_flat.csv"

# ------------------------------------------
# 1) extraer id + variables desde JSON
# ------------------------------------------
rows_stats = []

json_files = [f for f in os.listdir(JSON_DIR) if f.lower().endswith(".json")]

for fname in json_files:
    fpath = os.path.join(JSON_DIR, fname)

    try:
        with open(fpath, "r", encoding="utf-8") as f:
            data = json.load(f)

        obj = data.get("obj", data)
        cm_stats = obj.get("cm_statistics", {})

        rows_stats.append({
            "response_id": obj.get("id"),

            "num_sp_editorial_playlists": cm_stats.get("num_sp_editorial_playlists"),
            "num_sp_playlists": cm_stats.get("num_sp_playlists"),
            "sp_editorial_playlist_total_reach": cm_stats.get("sp_editorial_playlist_total_reach"),
            "num_am_editorial_playlists": cm_stats.get("num_am_editorial_playlists"),
            "num_am_playlists": cm_stats.get("num_am_playlists"),
            "num_de_editorial_playlists": cm_stats.get("num_de_editorial_playlists"),
            "num_de_playlists": cm_stats.get("num_de_playlists"),
            "de_playlist_total_reach": cm_stats.get("de_playlist_total_reach"),
            "de_editorial_playlist_total_reach": cm_stats.get("de_editorial_playlist_total_reach"),
            "num_az_editorial_playlists": cm_stats.get("num_az_editorial_playlists"),
            "num_az_playlists": cm_stats.get("num_az_playlists"),
            "num_yt_editorial_playlists": cm_stats.get("num_yt_editorial_playlists"),
            "num_yt_playlists": cm_stats.get("num_yt_playlists"),
            "yt_playlist_total_reach": cm_stats.get("yt_playlist_total_reach"),

            "tiktok_top_video_views": cm_stats.get("tiktok_top_video_views"),
            "tiktok_top_video_comments": cm_stats.get("tiktok_top_video_comments"),
            "tiktok_track_posts": cm_stats.get("tiktok_track_posts")
        })

    except Exception as e:
        print(f"Error leyendo {fname}: {e}")

df_stats = pd.DataFrame(rows_stats).drop_duplicates(subset=["response_id"])

# ------------------------------------------
# 2) leer metadata_flat
# ------------------------------------------
df_metadata = pd.read_csv(CSV_PATH)

# ------------------------------------------
# 3) merge limpio
# ------------------------------------------
cols_to_replace = [
    "num_sp_editorial_playlists",
    "num_sp_playlists",
    "sp_editorial_playlist_total_reach",
    "num_am_editorial_playlists",
    "num_am_playlists",
    "num_de_editorial_playlists",
    "num_de_playlists",
    "de_playlist_total_reach",
    "de_editorial_playlist_total_reach",
    "num_az_editorial_playlists",
    "num_az_playlists",
    "num_yt_editorial_playlists",
    "num_yt_playlists",
    "yt_playlist_total_reach",
    "tiktok_top_video_views",
    "tiktok_top_video_comments",
    "tiktok_track_posts"
]

df_metadata = df_metadata.drop(columns=cols_to_replace, errors="ignore")

df_metadata = df_metadata.merge(
    df_stats,
    on="response_id",
    how="left"
)

# ------------------------------------------
# 4) guardar
# ------------------------------------------
df_metadata.to_csv(CSV_PATH, index=False)

print("metadata_flat.csv actualizado con variables de playlists y TikTok")

for col in cols_to_replace:
    print(f"\n{col}")
    print(df_metadata[col].notna().sum(), "valores no nulos")

metadata_flat.csv actualizado con variables de playlists y TikTok

num_sp_editorial_playlists
1398 valores no nulos

num_sp_playlists
1398 valores no nulos

sp_editorial_playlist_total_reach
1398 valores no nulos

num_am_editorial_playlists
1384 valores no nulos

num_am_playlists
1384 valores no nulos

num_de_editorial_playlists
1368 valores no nulos

num_de_playlists
1368 valores no nulos

de_playlist_total_reach
1368 valores no nulos

de_editorial_playlist_total_reach
1368 valores no nulos

num_az_editorial_playlists
1350 valores no nulos

num_az_playlists
1350 valores no nulos

num_yt_editorial_playlists
1373 valores no nulos

num_yt_playlists
1373 valores no nulos

yt_playlist_total_reach
1369 valores no nulos

tiktok_top_video_views
1394 valores no nulos

tiktok_top_video_comments
1394 valores no nulos

tiktok_track_posts
1394 valores no nulos


# Merge con metadata

In [37]:
ruta_metadata = r"chartmetric_metadata\metadata_flat.csv"

df_metadata = pd.read_csv(ruta_metadata)

print(df_metadata.shape)
df_metadata.head()

(1400, 51)


In [38]:
df_metadata = df_metadata.rename(columns={
    "response_id": "chartmetric_id"
})

In [39]:
print("Duplicados en metadata:", df_metadata["chartmetric_id"].duplicated().sum())

Duplicados en metadata: 0


In [41]:
df_metadata = df_metadata.rename(columns={
    "name": "artist_name"
})

In [42]:
df_final = df_metadata.merge(
    df_live_agg,
    on="chartmetric_id",
    how="left",
    suffixes=("_meta", "_live")
)

In [45]:
df_final["validacion_requested_vs_meta"] = (
    df_final["requested_artist_name"] == df_final["artist_name_meta"]
)

df_final["validacion_meta_vs_live"] = (
    df_final["artist_name_live"].isna() |
    (df_final["artist_name_meta"] == df_final["artist_name_live"])
)

In [51]:
print("Filas metadata:", df_metadata.shape[0])
print("Filas final:", df_final.shape[0])
print("Duplicados en df_final por chartmetric_id:", df_final["chartmetric_id"].duplicated().sum())
print("Sin match en live:", df_final["artist_name_live"].isna().sum())

Filas metadata: 1400
Filas final: 1400
Duplicados en df_final por chartmetric_id: 0
Sin match en live: 396


# Dataset final

In [53]:
df_final["validacion_meta_vs_live"].value_counts()

validacion_meta_vs_live
True     1399
False       1
Name: count, dtype: int64

In [54]:
df_final.loc[
    df_final["validacion_meta_vs_live"] == False,
    ["chartmetric_id", "artist_name_meta", "artist_name_live"]
]

In [55]:
df_final.loc[
    df_final["chartmetric_id"] == 209368,
    ["artist_name_meta", "artist_name_live"]
] = ["Sofi Tukker", "Sofi Tukker"]

## Eliminación de columnas innecesarias

In [58]:
cols_drop = [
    "downloaded_at_utc",
    "requested_chartmetric_id",
    "requested_artist_name",
    "artist_name_live",
    "validacion_requested_vs_meta",
    "validacion_meta_vs_live"
]

df_final = df_final.drop(columns=cols_drop)

In [59]:
df_final = df_final.rename(columns={
    "artist_name_meta": "artist_name",
    "code2": "country"
})

In [60]:
print(df_final.shape)
df_final.head()

(1400, 77)


In [61]:
df_final.columns

Index(['chartmetric_id', 'artist_name', 'country', 'pronoun_title',
       'gender_title', 'record_label', 'hometown_city', 'cm_artist_rank',
       'cm_artist_score', 'career_stage', 'career_stage_score', 'career_trend',
       'career_trend_score', 'primary_genre', 'sp_followers',
       'sp_monthly_listeners', 'sp_popularity', 'sp_playlist_total_reach',
       'ins_followers', 'twitter_followers', 'tiktok_followers',
       'tiktok_likes', 'ycs_subscribers', 'ycs_views',
       'youtube_daily_video_views', 'youtube_monthly_video_views',
       'deezer_fans', 'shazam_count', 'pandora_lifetime_streams',
       'pandora_lifetime_stations_added', 'band', 'num_sp_editorial_playlists',
       'num_sp_playlists', 'sp_editorial_playlist_total_reach',
       'num_am_editorial_playlists', 'num_am_playlists',
       'num_de_editorial_playlists', 'num_de_playlists',
       'de_playlist_total_reach', 'de_editorial_playlist_total_reach',
       'num_az_editorial_playlists', 'num_az_playlists',
  

# Guardado de cohorte 1

In [62]:
df_final.to_csv("cohorte1_raw_1400.csv", index=False)